# Thucy: An LLM-based MAS for Claim Verification


![The architecture of Thucy](https://raw.githubusercontent.com/michaeltheologitis/thucy/main/docs/images/thucy.png)

<div align="center">

[![AAAI'26@LaMAS](https://img.shields.io/badge/paper-A42C25?style=for-the-badge&logo=arxiv&logoColor=white)](https://arxiv.org/abs/2512.03278) 

</div>

In today's age, it is becoming increasingly difficult to decipher truth from lies. Every day, politicians, media outlets, and public figures make conflicting claims—often about topics that can, in principle, be verified against structured data. For instance, statements about crime rates, economic growth or healthcare can all be verified against official public records and structured datasets. Building a system that can automatically do that would have sounded like science fiction just a few years ago. Yet, with the extraordinary progress in LLMs and agentic AI, this is now within reach. Still, there remains a striking gap between what is technically possible and what is being demonstrated by recent work. Most existing verification systems operate only on small, single-table databases—typically a few hundred rows—that conveniently fit within an LLM's context window.

In this project, we propose **Thucy**, the first cross-database, cross-table multi-agent claim verification system that also provides concrete evidence for each verification verdict. Thucy remains **completely agnostic to the underlying data sources** before deployment and must therefore autonomously discover, inspect, and reason over all available relational databases to verify claims. Importantly, Thucy also reports the exact SQL queries that support its verdict (whether the claim is accurate or not) offering full transparency to expert users familiar with SQL. When evaluated on the TabFact dataset—the standard benchmark for fact verification over structured data—Thucy surpasses the previous state of the art by 5.6 percentage points in accuracy (94.3% vs. 88.7%).


We highly encourage to read the docs from the official [Thucy website][docs] (complete, and better rendering).

[docs]: https://michaeltheologitis.github.io/thucy/

## Usage Example

Given the daily-updated Seattle [crime data](https://data.seattle.gov/Public-Safety/SPD-Crime-Data-2008-Present/tazs-3rd5/about_data) (1.5GB) from the City of Seattle, we set out to verify the claim:

> The number of violent crimes decreased in Seattle between 2024 and 2023.

We can simply load the data into a database and ask Thucy to verify the claim:

```sh 
thucy verify "The number of violent crimes decreased in Seattle between 2024 and 2023" --workflow "Violent-Crimes"
```

In [ ]:
#| notest
#| echo: false
#!thucy verify "The number of violent crimes decreased in Seattle between 2024 and 2023" --workflow "Violent-Crimes"
print(
"""Setting up connection to Google's toolbox...
Success!
Starting Verification. Might take a while...!
Success!

Thucy Verdict: Inaccurate

Saved full report to experiments/results/Violent-Crimes-trace_824464db51d741d196726bed0316f034.txt
Exiting!
"""
)

Setting up connection to Google's toolbox...
Success!
Starting Verification. Might take a while...!
Success!

Thucy Verdict: Inaccurate

Saved full report to experiments/results/Violent-Crimes-trace_824464db51d741d196726bed0316f034.txt
Exiting!



# Installation

## Clone

Clone from [GitHub][repo]:

```sh
git clone https://github.com/michaeltheologitis/thucy.git
```

Go to the project directory:

```sh
cd thucy
```

Then, install the package (usually in a virtual environment):

```sh
pip install -e .
```


[repo]: https://github.com/michaeltheologitis/thucy

## Configurations

You'll need to configure: ① OpenAI API key, ② toolsets for the agents, and ③ database connections in `tools.yaml`.

### OpenAI API Key

You need an OpenAI API key. Get one at [OpenAI's website](https://platform.openai.com/api-keys).

```sh
thucy config set OPENAI_API_KEY <your-api-key>
```

### Toolsets

Configure which toolsets each agent uses (these must match names in your `tools.yaml`):

```sh
thucy config set SQL_EXPERT_TOOLSET seattle-sql
thucy config set SCHEMA_EXPERT_TOOLSET seattle-schema
thucy config set DATA_EXPERT_TOOLSET seattle-schema
```

### Verify Configuration

```sh
thucy config show
```

In [ ]:
#| notest
#| echo: false
print("""
OPENAI_API_KEY=sk-xxyy
EXPERTS_MODEL=gpt-5-mini
LEAD_MODEL=gpt-5
LEAD_MAX_TURNS=40
SQL_EXPERT_MAX_TURNS=30
SCHEMA_EXPERT_MAX_TURNS=30
DATA_EXPERT_MAX_TURNS=30
GENAI_SERVER_URL=http://127.0.0.1:5000
SQL_EXPERT_TOOLSET=seattle-sql
SCHEMA_EXPERT_TOOLSET=seattle-schema
DATA_EXPERT_TOOLSET=seattle-schema
""")


OPENAI_API_KEY=sk-xxyy
EXPERTS_MODEL=gpt-5-mini
LEAD_MODEL=gpt-5
LEAD_MAX_TURNS=40
SQL_EXPERT_MAX_TURNS=30
SCHEMA_EXPERT_MAX_TURNS=30
DATA_EXPERT_MAX_TURNS=30
GENAI_SERVER_URL=http://127.0.0.1:5000
SQL_EXPERT_TOOLSET=seattle-sql
SCHEMA_EXPERT_TOOLSET=seattle-schema
DATA_EXPERT_TOOLSET=seattle-schema



### Google's MCP Toolbox

#### Databases

We use [Google's MCP Toolbox](https://cloud.google.com/blog/products/ai-machine-learning/mcp-toolbox-for-databases-now-supports-model-context-protocol) to manage database connections. Edit `tools.yaml` to configure your databases:

```yaml
sources:
  postgres-seattle:
    database: seattle
    host: localhost
    kind: postgres
    password: guest_pass
    port: 5432
    user: guest_user
```

Replace with your actual database credentials. See the [docs](https://googleapis.github.io/genai-toolbox/resources/sources/postgres/) for other database types (MySQL, SQLite, etc.).

#### Tools

Define tools that bind to your database sources:

```yaml
tools:
  postgres_seattle_execute_sql:
    description: Executes SQL queries on the PostgreSQL Seattle database.
    kind: postgres-execute-sql
    source: postgres-seattle
  postgres_seattle_list_tables:
    description: Retrieves schema information from the Seattle database.
    kind: postgres-list-tables
    source: postgres-seattle
```

#### Defining Toolsets

Finally, group your tools into **toolsets**:

```yaml
toolsets:
  seattle-schema:
  - postgres_seattle_list_tables
  seattle-sql:
  - postgres_seattle_execute_sql
```

These toolset names (`seattle-schema`, `seattle-sql`) are what you'll configure via `thucy config set` to tell each agent which tools to use.

#### Running the MCP Toolbox

Now, we are ready to run toolbox. Send it:

```sh
toolbox --ui
```

In [ ]:
#| notest
#| echo: false
print("""2025-12-02T19:36:39.07939-08:00 INFO "Initialized 1 sources." 
2025-12-02T19:36:39.079447-08:00 INFO "Initialized 0 authServices." 
2025-12-02T19:36:39.079637-08:00 INFO "Initialized 2 tools." 
2025-12-02T19:36:39.079648-08:00 INFO "Initialized 3 toolsets." 
2025-12-02T19:36:39.079846-08:00 INFO "Server ready to serve!" 
2025-12-02T19:36:39.079852-08:00 INFO "Toolbox UI is up and running at: http://127.0.0.1:5000/ui""")

2025-12-02T19:36:39.07939-08:00 INFO "Initialized 1 sources." 
2025-12-02T19:36:39.079447-08:00 INFO "Initialized 0 authServices." 
2025-12-02T19:36:39.079637-08:00 INFO "Initialized 2 tools." 
2025-12-02T19:36:39.079648-08:00 INFO "Initialized 3 toolsets." 
2025-12-02T19:36:39.079846-08:00 INFO "Server ready to serve!" 
2025-12-02T19:36:39.079852-08:00 INFO "Toolbox UI is up and running at: http://127.0.0.1:5000/ui


If you encounter errors, check for YAML indentation mistakes. See the [docs](https://googleapis.github.io/genai-toolbox/resources/sources/) for help.

## Add Data

Load data into your database(s). Examples: [Seattle Crime Data](https://data.seattle.gov/Public-Safety/SPD-Crime-Data-2008-Present/tazs-3rd5/about_data), [Los Angeles Crime](https://catalog.data.gov/dataset/crime-data-from-2020-to-present). Don't worry about messy data—Thucy handles that!

## Run Thucy

Start the toolbox server:

```sh
toolbox --ui
```

Then, run Thucy:

```sh
thucy verify "<CLAIM-TO-VERIFY>" --workflow "<CUSTOM-NAME>"
```

The `--workflow` flag is just a name to help identify your results file.

# Paper Results

In order to vizualise the results of the paper (or reproduce them from scratch), please run the notebook `experiments/paper/tabfact.ipynb`.

# Extend & Develop

This project has been created using [nbdev](https://nbdev.fast.ai/) (big shoutout!). 

If you want to develop independently first run:

```sh
nbdev_install_quarto
nbdev_install_hooks
```